In [11]:
import pandas as pd
import joblib
import json

# Load model and expected features
model = joblib.load("best_rf_qb_model.joblib")
with open("rf_feature_cols.json", "r") as f:
    feature_cols = json.load(f)

# Load prediction data
pred_df = pd.read_csv("../data/processed/qb_pred_combined.csv")

print("Prediction set shape:", pred_df.shape)
print("\nExpected feature cols:")
print(feature_cols)
print("\nPrediction set cols:")
print(pred_df.columns.tolist())

Prediction set shape: (932, 21)

Expected feature cols:
['overall', 'round', 'pick', 'height', 'weight', 'pre_draft_ranking', 'pre_draft_position_ranking', 'pre_draft_grade', 'completions_pg', 'attempts_pg', 'passing_yards_pg', 'passing_tds_pg', 'passing_interceptions_pg', 'carries_pg', 'rushing_yards_pg', 'rushing_tds_pg', 'rushing_fumbles_lost_pg', 'rushing_fumbles_pg', 'college_flag']

Prediction set cols:
['player_id', 'player_display_name', 'overall', 'round', 'pick', 'height', 'weight', 'pre_draft_ranking', 'pre_draft_position_ranking', 'pre_draft_grade', 'completions_pg', 'attempts_pg', 'passing_yards_pg', 'passing_tds_pg', 'passing_interceptions_pg', 'carries_pg', 'rushing_yards_pg', 'rushing_tds_pg', 'rushing_fumbles_lost_pg', 'rushing_fumbles_pg', 'college_flag']


In [12]:
# Separate identifiers from features
identifiers = pred_df[["player_id", "player_display_name", "college_flag"]]
X_pred = pred_df[feature_cols]

# Generate predictions
predictions = model.predict(X_pred)

# Attach predictions back to identifiers
results_df = identifiers.copy()
results_df["predicted_weekly_ppr"] = predictions.round(2)

# Sort by predicted score descending
results_df = results_df.sort_values("predicted_weekly_ppr", ascending=False).reset_index(drop=True)

# Preview top 20
print(results_df.head(20))

# Export
results_df.to_csv("../data/processed/qb_predictions_2026.csv", index=False)
print("\nExport complete")
print(f"Total players predicted: {len(results_df)}")
print(f"NFL players: {(results_df['college_flag'] == 0).sum()}")
print(f"CFB players: {(results_df['college_flag'] == 1).sum()}")

     player_id player_display_name  college_flag  predicted_weekly_ppr
0   00-0033873     Patrick Mahomes             0                 19.17
1   00-0034857          Josh Allen             0                 19.14
2   00-0039732              Bo Nix             0                 18.82
3   00-0039851          Drake Maye             0                 17.55
4   00-0036355      Justin Herbert             0                 17.32
5   00-0026498    Matthew Stafford             0                 17.12
6   00-0036971     Trevor Lawrence             0                 17.10
7   00-0039918      Caleb Williams             0                 16.53
8   00-0033077        Dak Prescott             0                 16.42
9   00-0036389         Jalen Hurts             0                 16.39
10  00-0035228        Kyler Murray             0                 16.04
11  00-0033106          Jared Goff             0                 15.99
12  00-0033119     Jacoby Brissett             0                 15.87
13  00